In [12]:
from google.colab import files

uploaded = files.upload()

Saving 선박제원정보_2024.csv to 선박제원정보_2024.csv
Saving 선박제원정보_2022.csv to 선박제원정보_2022.csv
Saving 선박제원정보_2021.csv to 선박제원정보_2021.csv


In [13]:
from google.colab import drive
drive.mount('/content/drive')

import os

base_path = "/content/drive/MyDrive/contest_changwon_port"

raw_path = os.path.join(base_path, "data/raw")
processed_path = os.path.join(base_path, "data/processed")

os.makedirs(raw_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

print("base_path:", base_path)
print("raw_path:", raw_path)
print("processed_path:", processed_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
base_path: /content/drive/MyDrive/contest_changwon_port
raw_path: /content/drive/MyDrive/contest_changwon_port/data/raw
processed_path: /content/drive/MyDrive/contest_changwon_port/data/processed


In [14]:
forwarder_folder = os.path.join(base_path, "data/raw/ship_spec")
os.makedirs(forwarder_folder, exist_ok=True)

print("선박제원 폴더:", forwarder_folder)

선박제원 폴더: /content/drive/MyDrive/contest_changwon_port/data/raw/ship_spec


In [15]:
import glob
import os

files = glob.glob(os.path.join(forwarder_folder, "*.csv"))

print("파일 수:", len(files))

for f in files:
    print(os.path.basename(f))

파일 수: 3
선박제원정보_2021.csv
선박제원정보_2022.csv
선박제원정보_2024.csv


In [16]:
import glob
import os
import unicodedata

# ship_spec_folder 안의 CSV 파일 찾기
files = glob.glob(os.path.join(ship_spec_folder, "*.csv"))

print("ship_spec 폴더 안 CSV 파일 수:", len(files))

for f in files:
    print(os.path.basename(f))

# 분석할 연도
target_years = ["2021", "2022", "2024"]

selected_files = []

for f in files:
    filename = os.path.basename(f)

    # 한글 파일명 자음/모음 분리 문제 해결
    filename_norm = unicodedata.normalize("NFC", filename)

    # 파일명에 2021, 2022, 2024 중 하나가 들어가면 선택
    if any(year in filename_norm for year in target_years):
        selected_files.append(f)

print("선택된 파일 수:", len(selected_files))

for f in selected_files:
    print(os.path.basename(f))

if len(selected_files) == 0:
    raise FileNotFoundError("선박제원정보 파일이 선택되지 않았습니다.")

ship_spec 폴더 안 CSV 파일 수: 3
선박제원정보_2021.csv
선박제원정보_2022.csv
선박제원정보_2024.csv
선택된 파일 수: 3
선박제원정보_2021.csv
선박제원정보_2022.csv
선박제원정보_2024.csv


In [17]:
import pandas as pd
import re

def read_file_korean(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print("읽기 성공:", os.path.basename(path), "/", enc)
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 실패: {path}")

In [18]:
# 데이터통합

df_list = []

for file in selected_files:
    df = read_file_korean(file)

    filename = os.path.basename(file)
    filename_norm = unicodedata.normalize("NFC", filename)

    # 파일명에서 연도 추출
    year_match = re.search(r"20\d{2}", filename_norm)
    if year_match:
        df["year"] = int(year_match.group())
    else:
        df["year"] = None

    df["source_file"] = filename_norm
    df_list.append(df)

ship_spec = pd.concat(df_list, ignore_index=True)

print("통합 데이터 크기:", ship_spec.shape)
display(ship_spec.head())
print(ship_spec.columns)

읽기 성공: 선박제원정보_2021.csv / utf-8-sig
읽기 성공: 선박제원정보_2022.csv / utf-8-sig
읽기 성공: 선박제원정보_2024.csv / utf-8-sig
통합 데이터 크기: (154691, 24)


,ibobprt,clsgn,vsslNo,imoNo,vsslKorNm,vsslEngNm,vsslKnd,vsslNlty,tonEdycSe,intrlGrtg,...,vsslDrft,vsslLt,vsslDp,brbtSe,nvgShapCd,nvgShapNm,vsslCnstrDt,befClsgn,year,source_file
0,1[외항],UCLA,NaN,NaN,MARO,MARO,92[원양 어선],RU[러시아],1.0,39.0,...,1.5,18.4,1.5,9.0,2.0,부정기선,1982-01-01T00:00:00+09:00,NaN,2021,선박제원정보_2021.csv
1,1[외항],XUGH9,NaN,NaN,STAR LINK,NaN,32[냉동.냉장선],KH[캄보디아],1.0,339.0,...,3.5,44.3,3.5,NaN,2.0,부정기선,1978-01-01T00:00:00+09:00,NaN,2021,선박제원정보_2021.csv
2,1[외항],#H9IO,NaN,NaN,WESTPACK EXPRESS,WESTPACK EXPRESS,83[군함],PA[파나마],1.0,8403.0,...,4.5,86.1,9.4,9.0,2.0,부정기선,2000-08-18T00:00:00+09:00,H9IO,2021,선박제원정보_2021.csv
3,1[외항],&V3VB6,NaN,NaN,OCEAN TUG NO.1,OCEAN TUG NO.1,69[기타 예선],BZ[벨리즈],1.0,346.0,...,4.0,30.0,4.0,NaN,2.0,부정기선,1975-01-01T00:00:00+09:00,V3VB6,2021,선박제원정보_2021.csv
4,1[외항],*3FMX,NaN,9130963,KESARI PREM,KESARI PREM,21[산물선(벌크선)],PA[파나마],1.0,36698.0,...,16.0,215.9,18.3,NaN,2.0,부정기선,2011-07-25T00:00:00+09:00,NaN,2021,선박제원정보_2021.csv


Index(['ibobprt', 'clsgn', 'vsslNo', 'imoNo', 'vsslKorNm', 'vsslEngNm',
       'vsslKnd', 'vsslNlty', 'tonEdycSe', 'intrlGrtg', 'grtg', 'ntng',
       'vsslTotLt', 'shdth', 'vsslDrft', 'vsslLt', 'vsslDp', 'brbtSe',
       'nvgShapCd', 'nvgShapNm', 'vsslCnstrDt', 'befClsgn', 'year',
       'source_file'],
      dtype='object')


In [19]:
# 컬럼명 앞뒤 공백 제거
ship_spec.columns = ship_spec.columns.astype(str).str.strip()

print("컬럼 목록:")
print(ship_spec.columns.tolist())

display(ship_spec.head())

컬럼 목록:
['ibobprt', 'clsgn', 'vsslNo', 'imoNo', 'vsslKorNm', 'vsslEngNm', 'vsslKnd', 'vsslNlty', 'tonEdycSe', 'intrlGrtg', 'grtg', 'ntng', 'vsslTotLt', 'shdth', 'vsslDrft', 'vsslLt', 'vsslDp', 'brbtSe', 'nvgShapCd', 'nvgShapNm', 'vsslCnstrDt', 'befClsgn', 'year', 'source_file']


,ibobprt,clsgn,vsslNo,imoNo,vsslKorNm,vsslEngNm,vsslKnd,vsslNlty,tonEdycSe,intrlGrtg,...,vsslDrft,vsslLt,vsslDp,brbtSe,nvgShapCd,nvgShapNm,vsslCnstrDt,befClsgn,year,source_file
0,1[외항],UCLA,NaN,NaN,MARO,MARO,92[원양 어선],RU[러시아],1.0,39.0,...,1.5,18.4,1.5,9.0,2.0,부정기선,1982-01-01T00:00:00+09:00,NaN,2021,선박제원정보_2021.csv
1,1[외항],XUGH9,NaN,NaN,STAR LINK,NaN,32[냉동.냉장선],KH[캄보디아],1.0,339.0,...,3.5,44.3,3.5,NaN,2.0,부정기선,1978-01-01T00:00:00+09:00,NaN,2021,선박제원정보_2021.csv
2,1[외항],#H9IO,NaN,NaN,WESTPACK EXPRESS,WESTPACK EXPRESS,83[군함],PA[파나마],1.0,8403.0,...,4.5,86.1,9.4,9.0,2.0,부정기선,2000-08-18T00:00:00+09:00,H9IO,2021,선박제원정보_2021.csv
3,1[외항],&V3VB6,NaN,NaN,OCEAN TUG NO.1,OCEAN TUG NO.1,69[기타 예선],BZ[벨리즈],1.0,346.0,...,4.0,30.0,4.0,NaN,2.0,부정기선,1975-01-01T00:00:00+09:00,V3VB6,2021,선박제원정보_2021.csv
4,1[외항],*3FMX,NaN,9130963,KESARI PREM,KESARI PREM,21[산물선(벌크선)],PA[파나마],1.0,36698.0,...,16.0,215.9,18.3,NaN,2.0,부정기선,2011-07-25T00:00:00+09:00,NaN,2021,선박제원정보_2021.csv


In [20]:
num_cols = [
    "intrlGrtg",    # 국제총톤수
    "grtg",         # 총톤수
    "ntng",         # 순톤수
    "vsslTotLt",    # 선박 전체 길이
    "shdth",        # 선박 폭
    "vsslDrft",     # 흘수
    "vsslLt",       # 선박 길이
    "vsslDp"        # 선박 깊이
]

exist_num_cols = [col for col in num_cols if col in ship_spec.columns]

for col in exist_num_cols:
    ship_spec[col] = (
        ship_spec[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("-", "0", regex=False)
        .str.strip()
    )
    ship_spec[col] = pd.to_numeric(ship_spec[col], errors="coerce")

print("숫자형으로 변환한 컬럼:")
print(exist_num_cols)

display(ship_spec[exist_num_cols].head())

숫자형으로 변환한 컬럼:
['intrlGrtg', 'grtg', 'ntng', 'vsslTotLt', 'shdth', 'vsslDrft', 'vsslLt', 'vsslDp']


,intrlGrtg,grtg,ntng,vsslTotLt,shdth,vsslDrft,vsslLt,vsslDp
0,39.0,39.0,13.0,18.4,NaN,1.5,18.4,1.5
1,339.0,339.0,209.0,44.3,NaN,3.5,44.3,3.5
2,8403.0,8403.0,2521.0,101.0,25.6,4.5,86.1,9.4
3,346.0,197.0,65.0,30.0,8.0,4.0,30.0,4.0
4,36698.0,36698.0,22980.0,215.9,32.2,16.0,215.9,18.3


In [21]:
# 선박종류 정리
if "vsslKnd" in ship_spec.columns:
    ship_spec["선박종류코드"] = ship_spec["vsslKnd"].astype(str).str.extract(r"^(\d+)")
    ship_spec["선박종류"] = ship_spec["vsslKnd"].astype(str).str.extract(r"\[(.*?)\]")

# 외항/내항 구분 정리
if "ibobprt" in ship_spec.columns:
    ship_spec["항행구분코드"] = ship_spec["ibobprt"].astype(str).str.extract(r"^(\d+)")
    ship_spec["항행구분"] = ship_spec["ibobprt"].astype(str).str.extract(r"\[(.*?)\]")

# 국적 정리
if "vsslNlty" in ship_spec.columns:
    ship_spec["국적코드"] = ship_spec["vsslNlty"].astype(str).str.extract(r"^([A-Z]+)")
    ship_spec["국적"] = ship_spec["vsslNlty"].astype(str).str.extract(r"\[(.*?)\]")

display_cols = [
    "year", "vsslKorNm", "선박종류", "항행구분", "국적",
    "grtg", "vsslTotLt", "shdth", "vsslDrft"
]

display_cols = [col for col in display_cols if col in ship_spec.columns]

display(ship_spec[display_cols].head())

,year,vsslKorNm,선박종류,항행구분,국적,grtg,vsslTotLt,shdth,vsslDrft
0,2021,MARO,원양 어선,외항,러시아,39.0,18.4,NaN,1.5
1,2021,STAR LINK,냉동.냉장선,외항,캄보디아,339.0,44.3,NaN,3.5
2,2021,WESTPACK EXPRESS,군함,외항,파나마,8403.0,101.0,25.6,4.5
3,2021,OCEAN TUG NO.1,기타 예선,외항,벨리즈,197.0,30.0,8.0,4.0
4,2021,KESARI PREM,산물선(벌크선),외항,파나마,36698.0,215.9,32.2,16.0


In [22]:
# 선박 이름 컬럼이 있으면 사용, 없으면 행 개수로 계산
# 연도별 선박 규모 요약


count_col = "vsslKorNm" if "vsslKorNm" in ship_spec.columns else ship_spec.columns[0]

agg_dict = {
    "선박수": (count_col, "count")
}

if "imoNo" in ship_spec.columns:
    agg_dict["고유_IMO수"] = ("imoNo", "nunique")

if "clsgn" in ship_spec.columns:
    agg_dict["고유_호출부호수"] = ("clsgn", "nunique")

if "grtg" in ship_spec.columns:
    agg_dict["평균총톤수"] = ("grtg", "mean")
    agg_dict["중앙값총톤수"] = ("grtg", "median")
    agg_dict["최대총톤수"] = ("grtg", "max")

if "vsslTotLt" in ship_spec.columns:
    agg_dict["평균길이"] = ("vsslTotLt", "mean")

if "shdth" in ship_spec.columns:
    agg_dict["평균폭"] = ("shdth", "mean")

if "vsslDrft" in ship_spec.columns:
    agg_dict["평균흘수"] = ("vsslDrft", "mean")

ship_year_summary = (
    ship_spec
    .groupby("year", as_index=False)
    .agg(**agg_dict)
)

display(ship_year_summary)

,year,선박수,고유_IMO수,고유_호출부호수,평균총톤수,중앙값총톤수,최대총톤수,평균길이,평균폭,평균흘수
0,2021,250,129,250,18706.244980,7316.0,160930.0,136.520496,22.396652,9.060169
1,2022,76663,36305,77283,24062.159636,11078.0,999999.0,144.496572,25.128315,9.692229
2,2024,77156,39082,77158,25316.593045,12743.0,999999.0,148.226632,25.267392,9.628589


In [25]:
# 연도별 상위 10개 선박 종류

ship_kind_top10 = (
    ship_kind_summary
    .groupby("year")
    .head(10)
)

display(ship_kind_top10)

,year,선박종류,선박수,평균총톤수,중앙값총톤수,최대총톤수,평균길이,평균폭,평균흘수
19,2021,일반화물선,58,13640.086207,7170.5,87709.0,134.252632,21.212281,9.337500
10,2021,산물선(벌크선),31,42647.225806,36698.0,110352.0,213.358065,33.623333,13.296774
11,2021,석유제품 운반선,25,21301.680000,5373.0,159423.0,159.968000,23.836000,9.784000
16,2021,원양 어선,21,1788.904762,759.0,14289.0,61.642857,11.165000,5.176190
24,2021,풀컨테이너선,20,31608.000000,18401.5,95138.0,199.168000,29.155000,10.615000
22,2021,케미칼 운반선,14,11284.214286,8571.5,29456.0,126.164286,21.100000,8.435714
7,2021,기타선,12,723.083333,83.5,7805.0,36.800000,8.857143,3.800000
8,2021,냉동.냉장선,11,4722.818182,4402.0,13575.0,104.354545,16.220000,7.072727
1,2021,견인용예선,7,94.857143,51.0,295.0,23.328571,6.925000,2.900000
12,2021,세미(혼재)컨테이너선,7,9780.571429,12220.0,14348.0,138.000000,21.337143,8.957143


In [26]:
# 총톤수 구간별 선박 수

import numpy as np

bins = [0, 500, 3000, 10000, 50000, np.inf]
labels = [
    "500톤 미만",
    "500~3,000톤",
    "3,000~10,000톤",
    "10,000~50,000톤",
    "50,000톤 이상"
]

ship_spec["톤급구간"] = pd.cut(
    ship_spec["grtg"],
    bins=bins,
    labels=labels,
    right=False
)

ton_class_summary = (
    ship_spec
    .groupby(["year", "톤급구간"], as_index=False, observed=False)
    .agg(
        선박수=(count_col, "count")
    )
)

ton_class_summary["비중(%)"] = (
    ton_class_summary["선박수"] /
    ton_class_summary.groupby("year")["선박수"].transform("sum") * 100
).round(1)

display(ton_class_summary)

,year,톤급구간,선박수,비중(%)
0,2021,500톤 미만,48,19.3
1,2021,"500~3,000톤",36,14.5
2,2021,"3,000~10,000톤",57,22.9
3,2021,"10,000~50,000톤",82,32.9
4,2021,"50,000톤 이상",26,10.4
5,2022,500톤 미만,15744,20.5
6,2022,"500~3,000톤",10732,14.0
7,2022,"3,000~10,000톤",10920,14.2
8,2022,"10,000~50,000톤",28927,37.7
9,2022,"50,000톤 이상",10339,13.5


In [27]:
# 항행구분별 분석

navigation_summary = (
    ship_spec
    .groupby(["year", "항행구분"], as_index=False)
    .agg(
        선박수=(count_col, "count"),
        평균총톤수=("grtg", "mean"),
        중앙값총톤수=("grtg", "median"),
        평균길이=("vsslTotLt", "mean")
    )
)

navigation_summary["비중(%)"] = (
    navigation_summary["선박수"] /
    navigation_summary.groupby("year")["선박수"].transform("sum") * 100
).round(1)

display(navigation_summary)

,year,항행구분,선박수,평균총톤수,중앙값총톤수,평균길이,비중(%)
0,2021,,1,NaN,NaN,NaN,0.4
1,2021,내항,26,1672.884615,184.5,52.980000,10.4
2,2021,외항,222,20785.351351,8589.5,146.761389,88.8
3,2021,항내운항,1,12.000000,12.0,13.000000,0.4
4,2022,,112,384.069107,31.0,35.725161,0.1
5,2022,내항,9698,1225.608158,54.0,39.361081,12.7
6,2022,외항,65826,27835.056915,16980.0,159.423215,85.9
7,2022,항내운항,1027,462.428768,49.0,27.484929,1.3
8,2024,,58,4843.241379,498.5,70.646667,0.1
9,2024,내항,7542,2069.025082,172.0,48.691304,9.8


In [28]:
# 국적별 상위 10개 분석

nationality_summary = (
    ship_spec
    .groupby(["year", "국적"], as_index=False)
    .agg(
        선박수=(count_col, "count"),
        평균총톤수=("grtg", "mean"),
        중앙값총톤수=("grtg", "median")
    )
    .sort_values(["year", "선박수"], ascending=[True, False])
)

nationality_top10 = (
    nationality_summary
    .groupby("year")
    .head(10)
)

display(nationality_top10)

,year,국적,선박수,평균총톤수,중앙값총톤수
31,2021,파나마,82,25910.365854,17599.0
3,2021,대한민국,64,8887.062500,953.0
8,2021,러시아,17,4375.647059,759.0
25,2021,중국,12,9088.750000,3633.0
16,2021,벨리즈,9,994.000000,1178.0
20,2021,싱가포르,9,33494.555556,29174.0
11,2021,마샬 제도,8,39221.625000,19290.0
7,2021,라이베리아,7,41557.428571,37949.0
34,2021,홍콩,6,13013.166667,12220.0
27,2021,캄보디아,4,887.250000,838.0


In [29]:
# 물류 관련 선박 후보 추출

cargo_keywords = [
    "화물",
    "컨테이너",
    "냉동",
    "냉장",
    "유조",
    "케미칼",
    "LPG",
    "LNG",
    "산적",
    "벌크",
    "자동차",
    "시멘트",
    "철강"
]

pattern = "|".join(cargo_keywords)

ship_spec["물류관련선박후보"] = ship_spec["선박종류"].astype(str).str.contains(
    pattern,
    case=False,
    na=False
)

cargo_ship_summary = (
    ship_spec
    .groupby(["year", "물류관련선박후보"], as_index=False)
    .agg(
        선박수=(count_col, "count"),
        평균총톤수=("grtg", "mean"),
        중앙값총톤수=("grtg", "median"),
        최대총톤수=("grtg", "max"),
        평균길이=("vsslTotLt", "mean"),
        평균폭=("shdth", "mean"),
        평균흘수=("vsslDrft", "mean")
    )
)

display(cargo_ship_summary)

,year,물류관련선박후보,선박수,평균총톤수,중앙값총톤수,최대총톤수,평균길이,평균폭,평균흘수
0,2021,False,98,13985.608247,979.0,160930.0,102.708791,18.301190,7.216279
1,2021,True,152,21718.756579,12962.0,110352.0,156.897086,24.705503,10.117333
2,2022,False,29678,18324.252292,699.0,999999.0,102.910503,22.071397,8.030699
3,2022,True,46985,27686.621742,19580.0,999999.0,169.423258,26.731095,10.608209
4,2024,False,28119,20066.771366,891.0,999999.0,107.684433,22.157808,7.760671
5,2024,True,49037,28327.181691,19918.0,999999.0,171.000665,26.864068,10.623328


In [30]:
ship_year_report_table = ship_year_summary.copy()

# 소수점 정리
for col in ship_year_report_table.columns:
    if ship_year_report_table[col].dtype in ["float64", "float32"]:
        ship_year_report_table[col] = ship_year_report_table[col].round(2)

display(ship_year_report_table)

,year,선박수,고유_IMO수,고유_호출부호수,평균총톤수,중앙값총톤수,최대총톤수,평균길이,평균폭,평균흘수
0,2021,250,129,250,18706.24,7316.0,160930.0,136.52,22.40,9.06
1,2022,76663,36305,77283,24062.16,11078.0,999999.0,144.50,25.13,9.69
2,2024,77156,39082,77158,25316.59,12743.0,999999.0,148.23,25.27,9.63


In [31]:
ship_kind_report_table = ship_kind_top10.copy()

for col in ship_kind_report_table.columns:
    if ship_kind_report_table[col].dtype in ["float64", "float32"]:
        ship_kind_report_table[col] = ship_kind_report_table[col].round(2)

display(ship_kind_report_table)

,year,선박종류,선박수,평균총톤수,중앙값총톤수,최대총톤수,평균길이,평균폭,평균흘수
19,2021,일반화물선,58,13640.09,7170.5,87709.0,134.25,21.21,9.34
10,2021,산물선(벌크선),31,42647.23,36698.0,110352.0,213.36,33.62,13.30
11,2021,석유제품 운반선,25,21301.68,5373.0,159423.0,159.97,23.84,9.78
16,2021,원양 어선,21,1788.90,759.0,14289.0,61.64,11.16,5.18
24,2021,풀컨테이너선,20,31608.00,18401.5,95138.0,199.17,29.16,10.62
22,2021,케미칼 운반선,14,11284.21,8571.5,29456.0,126.16,21.10,8.44
7,2021,기타선,12,723.08,83.5,7805.0,36.80,8.86,3.80
8,2021,냉동.냉장선,11,4722.82,4402.0,13575.0,104.35,16.22,7.07
1,2021,견인용예선,7,94.86,51.0,295.0,23.33,6.92,2.90
12,2021,세미(혼재)컨테이너선,7,9780.57,12220.0,14348.0,138.00,21.34,8.96


In [32]:
processed_path = os.path.join(base_path, "data/processed")
os.makedirs(processed_path, exist_ok=True)

ship_spec.to_csv(
    os.path.join(processed_path, "ship_spec_processed_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

ship_year_summary.to_csv(
    os.path.join(processed_path, "ship_spec_year_summary_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

ship_kind_summary.to_csv(
    os.path.join(processed_path, "ship_spec_kind_summary_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

ship_kind_top10.to_csv(
    os.path.join(processed_path, "ship_spec_kind_top10_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

ton_class_summary.to_csv(
    os.path.join(processed_path, "ship_spec_ton_class_summary_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

navigation_summary.to_csv(
    os.path.join(processed_path, "ship_spec_navigation_summary_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

nationality_top10.to_csv(
    os.path.join(processed_path, "ship_spec_nationality_top10_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

cargo_ship_summary.to_csv(
    os.path.join(processed_path, "ship_spec_cargo_candidate_summary_2021_2022_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("선박제원정보 분석 결과 저장 완료")
print("저장 위치:", processed_path)

선박제원정보 분석 결과 저장 완료
저장 위치: /content/drive/MyDrive/contest_changwon_port/data/processed


2022년과 2024년을 비교하면 평균총톤수, 중앙값총톤수, 평균길이가 증가, 이는 등록 선박의 규모가 전반적으로 커지는 경향을 보이는 것으로 해석 가능

선박종류별 분석 결과, 일반화물선, 산물선(벌크선), 풀컨테이너선, 석유제품 운반선, 원유운반선, 케미칼 운반선 등이 주요 선박 유형으로 나타남. 이는 분석 대상 선박들이 단순 여객 중심이 아니라 화물 운송 및 항만 물류와 관련된 선박 유형을 상당 부분 포함하고 있음을 보여줌

선박제원정보에서는 일반화물선, 벌크선, 컨테이너선, 석유제품 운반선 등 다양한 화물 관련 선박 유형이 확인됨. 이는 항만에서 처리되는 화물의 종류와 선박 규모가 다양할 수 있음을 보여주며, 항만시설과 배후 육상운송 인프라가 다양한 화물 운송 수요에 대응할 필요가 있음을 시사함


최대총톤수에는 999,999톤과 같은 극단값이 포함되어 있어, 선박 규모 해석에는 평균값과 함께 중앙값을 중심으로 활용


-> 선박제원정보 분석 결과, 2022년과 2024년에는 약 7만 6천 척 이상의 선박 정보가 확인되었으며, 2024년에는 2022년에 비해 평균총톤수와 중앙값총톤수, 평균 선박 길이가 증가한 것으로 나타났다. 선박종류별로는 일반화물선, 산물선(벌크선), 풀컨테이너선, 석유제품 운반선, 원유운반선, 케미칼 운반선 등이 주요 유형으로 확인되었다. 이는 항만 물류와 관련된 다양한 선박 규모와 화물 운송 유형이 존재함을 보여준다. 다만 선박제원정보는 특정 항만 입출항 실적과 직접 연결되는 자료가 아니므로, 마산항·진해항에 실제 입항한 선박으로 해석하지 않고 항만시설 분석을 보완하기 위한 선박 규모 특성 자료로 활용

=> 선박제원정보는 창원 항만에 실제 입항한 선박을 보여주는 자료는 아니지만, 항만시설 분석과 같은 연도에서 선박의 규모와 물류 관련 선박 유형을 확인함으로써 항만 배후 운송 인프라 분석을 보완하는 자료로 활용 가능